# RLS Audit notebook
**From the free guide *Who Sees What: a practical guide to Row-Level Security in Power BI* · DATASANJ (datasanj.com)**

This notebook checks row-level security for **every semantic model in one workspace**, in one run:

| Layer | What you get |
|---|---|
| 1. Workspace roles | Every user and group, with Admin, Member and Contributor marked **skips RLS** |
| 3. Role rules | Every role, the tables it filters, its rule and hidden columns (OLS) |
| 3. Leaky tables | For each role, the tables its filter never reaches |
| 4. Role members | Every member of every role |
| Testing | A DAX query run as each person on your test list, with PASS or FAIL |

**It only reads.** It changes no model, role or permission.

**How to use it:** attach a lakehouse (left pane → Lakehouses → Add), fill in the **Settings** cell below, then **Run all**.

**Needs:** a workspace on Fabric or Premium capacity, Admin or Member access to it, and Build permission on the models.
Testing as other people also needs the tenant setting *Dataset Execute Queries REST API*.


In [ ]:
%pip install -q semantic-link-labs

In [ ]:
# ---------- Settings: change these ----------
WORKSPACE = "Sales - Prod"          # the workspace to audit
MODELS = []                         # leave empty for every semantic model, or list names: ["Sales"]
SAVE_TO_LAKEHOUSE = True            # saves each run to the attached lakehouse

# Optional: test what real people see. One row per test.
# expected can be a number, ">0", "0" or "blank" (no rows).
TEST_USERS = [
    # {"model": "Sales", "user": "bob@contoso.com", "dax": 'EVALUATE ROW("v", [Total Sales])', "expected": "11400"},
    # {"model": "Sales", "user": "guest@partner.com", "dax": 'EVALUATE ROW("v", COUNTROWS(Sales))', "expected": "blank"},
]

# Tables that are fine to leave unfiltered, such as date tables
SAFE_TABLES = ["Date", "Calendar", "DimDate"]

In [ ]:
# ---------- RLS logic (no Fabric needed for this cell) ----------
import re
from collections import defaultdict

BYPASS_ROLES = {"Admin", "Member", "Contributor"}


def security_edges(rel):
    """Directions a security filter can travel across one relationship."""
    if not rel["active"]:
        return []
    one_to_one = rel["from_card"] == "One" and rel["to_card"] == "One"
    both = rel["cross"] == "BothDirections" and rel["security"] == "BothDirections"
    edges = [(rel["to"], rel["from"])]            # one side -> many side
    if both or one_to_one:
        edges.append((rel["from"], rel["to"]))    # many side -> one side
    return edges


def reached_tables(relationships, start_tables):
    graph = defaultdict(set)
    for rel in relationships:
        for a, b in security_edges(rel):
            graph[a].add(b)
    seen, todo = set(start_tables), list(start_tables)
    while todo:
        for nxt in graph[todo.pop()]:
            if nxt not in seen:
                seen.add(nxt)
                todo.append(nxt)
    return seen


def is_false(rule):
    return bool(re.fullmatch(r"\s*FALSE\s*\(\s*\)\s*", rule or "", re.I))


def model_findings(model, roles, relationships, data_tables, dq_tables, safe_tables):
    """roles: {name: {"filters": {table: rule}, "hidden": [..], "members": [..]}}"""
    out = []

    def add(level, part, role, msg):
        out.append({"Model": model, "Level": level, "Part": part, "Role": role, "Finding": msg})

    rls = {n: r for n, r in roles.items() if r["filters"]}
    ols = {n: r for n, r in roles.items() if r["hidden"]}
    if not roles:
        add("HIGH", 1, "", "No roles. Everyone who can open the report sees every row.")
    for n, r in roles.items():
        if not r["filters"] and not r["hidden"]:
            add("HIGH", 19, n, "Role has no rules, so its members see every row.")
        if not r["members"]:
            add("INFO", 17, n, "Role has no members. Users not in any role see no data.")
        people = [m for m in r["members"] if "@" in m]
        if people:
            add("INFO", 19, n, f"{len(people)} individual people are members. Security groups are easier to manage.")
    for n, r in rls.items():
        blocked = {t for t, f in r["filters"].items() if is_false(f)}
        reached = reached_tables(relationships, [t for t in r["filters"] if t not in blocked]) | blocked
        for t in sorted(set(data_tables) - reached):
            if t in safe_tables:
                continue
            add("HIGH", 2, n, f"Table '{t}' is not filtered by this role. Its members see all its rows.")
        rules = " ".join(r["filters"].values()).upper()
        if "USERNAME()" in rules:
            add("MEDIUM", 5, n, "Uses USERNAME(). USERPRINCIPALNAME() is the clearer choice for email matching.")
        for t, f in r["filters"].items():
            if t in dq_tables and re.search(r"\b(PATH\w*|CUSTOMDATA)\s*\(", f, re.I):
                add("HIGH", 15, n, f"PATH/CUSTOMDATA on DirectQuery table '{t}' won't work in a role rule.")
    rls_only = [n for n in rls if n not in ols]
    ols_only = [n for n in ols if n not in rls]
    if rls_only and ols_only:
        add("HIGH", 14, ", ".join(ols_only),
            f"OLS-only roles exist next to RLS-only roles ({', '.join(rls_only)}). Anyone in one of each gets visual errors.")
    return out


def passes(value, expected):
    expected = str(expected).strip().lower()
    if expected == "blank":
        return value in (None, "", 0)
    if value is None:
        return expected == "0"
    if expected == ">0":
        return float(value) > 0
    return abs(float(value) - float(expected)) < 0.005


In [ ]:
# ---------- Layer 1: workspace roles (who skips RLS) ----------
import pandas as pd
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

users = labs.list_workspace_users(workspace=WORKSPACE)
users["Skips RLS"] = users["Role"].isin(BYPASS_ROLES)
print(f"{int(users['Skips RLS'].sum())} of {len(users)} workspace members skip RLS (Admin, Member, Contributor).")
display(users[["User Name", "Email Address", "Type", "Role", "Skips RLS"]].sort_values("Skips RLS", ascending=False))

In [ ]:
# ---------- Layers 3 and 4: roles, rules, members and leaky tables ----------
models = fabric.list_datasets(workspace=WORKSPACE)["Dataset Name"].tolist()
if MODELS:
    models = [m for m in models if m in MODELS]

role_rows, member_rows, findings = [], [], []
for model in models:
    try:
        with connect_semantic_model(dataset=model, workspace=WORKSPACE, readonly=True) as tom:
            m = tom.model
            data_tables, dq_tables = [], set()
            for t in m.Tables:
                if any(str(c.Type) != "RowNumber" for c in t.Columns):
                    data_tables.append(t.Name)
                if any(str(p.Mode) in ("DirectQuery", "Dual") for p in t.Partitions):
                    dq_tables.add(t.Name)
            rels = [{"from": r.FromTable.Name, "to": r.ToTable.Name,
                     "from_card": str(r.FromCardinality), "to_card": str(r.ToCardinality),
                     "cross": str(r.CrossFilteringBehavior), "security": str(r.SecurityFilteringBehavior),
                     "active": bool(r.IsActive)} for r in m.Relationships]
            roles = {}
            for role in m.Roles:
                entry = {"filters": {}, "hidden": [], "members": [mm.MemberName for mm in role.Members]}
                for tp in role.TablePermissions:
                    if tp.FilterExpression:
                        entry["filters"][tp.Table.Name] = tp.FilterExpression
                    if str(tp.MetadataPermission) == "None":
                        entry["hidden"].append(tp.Table.Name)
                    for cp in tp.ColumnPermissions:
                        if str(cp.MetadataPermission) == "None":
                            entry["hidden"].append(f"{tp.Table.Name}[{cp.Name}]")
                    role_rows.append({"Model": model, "Role": role.Name, "Table": tp.Table.Name,
                                      "Rule": tp.FilterExpression or "", "Hidden (OLS)": ", ".join(
                                          [cp.Name for cp in tp.ColumnPermissions if str(cp.MetadataPermission) == "None"])})
                for name in entry["members"]:
                    member_rows.append({"Model": model, "Role": role.Name, "Member": name})
                roles[role.Name] = entry
            findings += model_findings(model, roles, rels, data_tables, dq_tables, set(SAFE_TABLES))
    except Exception as e:
        findings.append({"Model": model, "Level": "ERROR", "Part": "", "Role": "", "Finding": f"Could not read model: {e}"})

roles_df = pd.DataFrame(role_rows, columns=["Model", "Role", "Table", "Rule", "Hidden (OLS)"])
members_df = pd.DataFrame(member_rows, columns=["Model", "Role", "Member"])
findings_df = pd.DataFrame(findings, columns=["Model", "Level", "Part", "Role", "Finding"])
order = {"ERROR": 0, "HIGH": 1, "MEDIUM": 2, "INFO": 3}
findings_df = findings_df.sort_values(by="Level", key=lambda s: s.map(order))

print(f"Checked {len(models)} model(s): {(findings_df.Level == 'HIGH').sum()} high findings.")
display(findings_df)
display(roles_df)
display(members_df)

In [ ]:
# ---------- Testing: what does each person see? ----------
test_rows = []
for t in TEST_USERS:
    try:
        df = labs.evaluate_dax_impersonation(dataset=t["model"], dax_query=t["dax"], user_name=t["user"], workspace=WORKSPACE)
        value = None if df.empty else df.iloc[0, 0]
        ok, error = passes(value, t["expected"]), ""
    except Exception as e:
        value, ok, error = None, False, str(e)[:200]
    test_rows.append({"Model": t["model"], "User": t["user"], "Expected": t["expected"],
                      "Got": "(no rows)" if value is None else value, "Result": "PASS" if ok else "FAIL", "Error": error})

tests_df = pd.DataFrame(test_rows, columns=["Model", "User", "Expected", "Got", "Result", "Error"])
if TEST_USERS:
    print(f"{(tests_df.Result == 'PASS').sum()} passed, {(tests_df.Result == 'FAIL').sum()} failed.")
    display(tests_df)
else:
    print("No test users set. Add some to TEST_USERS in the Settings cell.")

In [ ]:
# ---------- Save this run, so you can compare with last month ----------
from datetime import datetime, timezone

run = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
if SAVE_TO_LAKEHOUSE:
    for name, df in {"rls_audit_workspace_users": users, "rls_audit_roles": roles_df, "rls_audit_members": members_df,
                     "rls_audit_findings": findings_df, "rls_audit_tests": tests_df}.items():
        if len(df):
            out = df.copy().astype(str)
            out.columns = [c.replace(" ", "_").replace("(", "").replace(")", "") for c in out.columns]
            out.insert(0, "Workspace", WORKSPACE)
            out.insert(0, "Run", run)
            spark.createDataFrame(out).write.mode("append").option("mergeSchema", "true").saveAsTable(name)
    print(f"Saved run {run} to the attached lakehouse (tables starting rls_audit_).")

## Compare with last month
Each run adds rows with a **Run** date. In the lakehouse SQL endpoint, a query like this shows members added since the last run:

```sql
SELECT Model, Role, Member FROM rls_audit_members WHERE Run = (SELECT MAX(Run) FROM rls_audit_members)
EXCEPT
SELECT Model, Role, Member FROM rls_audit_members WHERE Run = (SELECT MAX(Run) FROM rls_audit_members WHERE Run < (SELECT MAX(Run) FROM rls_audit_members))
```

Swap the two halves to see who was removed. Guide and video: **datasanj.com/blog-rls-guide.html**
